# Task-wise comparison

Paired, task-by-task comparison of all five conditions on both benchmarks. Every
condition runs the same tasks, so success/failure is compared per (task, fold)
instance rather than in aggregate. Significance comes from the exact McNemar test
on the discordant pairs.

Four tables: GAIA and OfficeBench, each with rich and sparse agent cards.

In [1]:
import sys
from itertools import combinations
from pathlib import Path

analysis_dir = Path.cwd() if (Path.cwd() / "metrics").exists() else Path.cwd() / "analysis"
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import pandas as pd
from scipy.stats import binomtest

from metrics.conditions import GAIA_CONFIG, OFFICEBENCH_CONFIG, load_card

In [2]:
CONDITIONS = ["BL-Lower", "BL-Upper", "Blueprint", "Playbook", "Adaptive System"]


def mcnemar_p(runs, condition_a, condition_b):
    """Exact McNemar p-value over the (task, fold) instances both conditions ran."""
    keyed = runs.assign(key=runs["task_key"].astype(str) + "::" + runs["fold"].astype(str))
    a = keyed[keyed["condition"] == condition_a].set_index("key")["is_success"]
    b = keyed[keyed["condition"] == condition_b].set_index("key")["is_success"]
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]

    a_only = int(((a == 1) & (b == 0)).sum())
    b_only = int(((a == 0) & (b == 1)).sum())
    discordant = a_only + b_only
    if not discordant:
        return 1.0
    return binomtest(min(a_only, b_only), discordant, 0.5).pvalue


def pairwise_pvalues(runs, conditions=CONDITIONS):
    """Symmetric matrix of McNemar p-values on overall success."""
    matrix = pd.DataFrame("", index=conditions, columns=conditions)
    for x, y in combinations(conditions, 2):
        matrix.loc[x, y] = matrix.loc[y, x] = round(mcnemar_p(runs, x, y), 4)
    for condition in conditions:
        matrix.loc[condition, condition] = "-"
    return matrix

In [3]:
for card in ["rich", "sparse"]:
    df = load_card(card, GAIA_CONFIG)
    print(f"=== GAIA {card} — McNemar p (overall success); no pair is significant if all > 0.05 ===")
    display(pairwise_pvalues(df))

=== GAIA rich — McNemar p (overall success); no pair is significant if all > 0.05 ===


,BL-Lower,BL-Upper,Blueprint,Playbook,Adaptive System
BL-Lower,-,0.1143,0.8043,1.0,0.5386
BL-Upper,0.1143,-,0.248,0.148,0.4222
Blueprint,0.8043,0.248,-,0.905,0.8176
Playbook,1.0,0.148,0.905,-,0.6353
Adaptive System,0.5386,0.4222,0.8176,0.6353,-


=== GAIA sparse — McNemar p (overall success); no pair is significant if all > 0.05 ===


,BL-Lower,BL-Upper,Blueprint,Playbook,Adaptive System
BL-Lower,-,0.3135,0.8939,0.8072,0.905
BL-Upper,0.3135,-,0.1849,0.5446,0.4887
Blueprint,0.8939,0.1849,-,0.5831,0.694
Playbook,0.8072,0.5446,0.5831,-,1.0
Adaptive System,0.905,0.4887,0.694,1.0,-


In [4]:
for card in ["rich", "sparse"]:
    df = load_card(card, OFFICEBENCH_CONFIG)
    print(f"=== OFFICEBENCH {card} — McNemar p (overall success) ===")
    display(pairwise_pvalues(df))

=== OFFICEBENCH rich — McNemar p (overall success) ===


,BL-Lower,BL-Upper,Blueprint,Playbook,Adaptive System
BL-Lower,-,0.0,0.0,0.0,0.0
BL-Upper,0.0,-,0.0505,0.5095,0.1898
Blueprint,0.0,0.0505,-,0.0072,0.6057
Playbook,0.0,0.5095,0.0072,-,0.0167
Adaptive System,0.0,0.1898,0.6057,0.0167,-


=== OFFICEBENCH sparse — McNemar p (overall success) ===


,BL-Lower,BL-Upper,Blueprint,Playbook,Adaptive System
BL-Lower,-,0.0001,0.0,0.0,0.0
BL-Upper,0.0001,-,0.0,0.0057,0.0002
Blueprint,0.0,0.0,-,0.3673,0.7894
Playbook,0.0,0.0057,0.3673,-,0.2065
Adaptive System,0.0,0.0002,0.7894,0.2065,-
